# 03 — Evaluating the compliance auditor

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/04_compliance_auditor/03_evaluate.ipynb)

**Prerequisites**:
- [02_build.ipynb](./02_build.ipynb)

**What you will learn**:
- How to build an evaluation framework for compliance automation
- Measuring requirement extraction completeness (precision/recall)
- Calibrating coverage scores against expert ground truth
- Analysing gap detection error rates and their cost implications
- Using LLM-as-judge for remediation quality assessment
- End-to-end performance: time, cost, and accuracy vs. manual audits

In [ ]:
# @title Setup — Run this cell first
!pip install -q "openai>=1.0.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "numpy>=1.24.0" "scikit-learn>=1.2.0"

import os, json, textwrap
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, cohen_kappa_score, mean_absolute_error,
    precision_score, recall_score, f1_score, roc_curve, auc,
    ConfusionMatrixDisplay,
)

from openai import OpenAI

client: Optional[OpenAI] = None
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    client = OpenAI(api_key=api_key)
    print("OpenAI client initialised ✓")
else:
    print("⚠ OPENAI_API_KEY not set — LLM-as-judge calls will use mock scores")

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — Evaluation framework

We evaluate the compliance auditor across **five dimensions**:

| Dimension | Metric | Why it matters |
|-----------|--------|----------------|
| Extraction completeness | Precision / Recall | Missing a requirement = undetected risk |
| Coverage scoring accuracy | Cohen's κ, MAE | Calibrated scores drive correct prioritisation |
| Gap detection (false gaps) | False positive rate | False alarms waste remediation effort |
| Gap detection (missed gaps) | False negative rate | Missed gaps = regulatory exposure |
| Remediation quality | LLM-as-judge (1-5) | Bad advice is worse than no advice |

In [ ]:
# ── Framework config ──
@dataclass
class EvalConfig:
    """Configuration for evaluation runs."""
    coverage_threshold: int = 3      # score < threshold → "gap"
    gap_cost_fp: float = 5_000.0     # cost of false positive (wasted remediation)
    gap_cost_fn: float = 50_000.0    # cost of false negative (regulatory fine risk)

config = EvalConfig()
print(f"Eval config:")
print(f"  Gap threshold  : score < {config.coverage_threshold} → flagged as gap")
print(f"  FP cost        : ${config.gap_cost_fp:,.0f} per false alarm")
print(f"  FN cost        : ${config.gap_cost_fn:,.0f} per missed gap")
print(f"  FN/FP ratio    : {config.gap_cost_fn / config.gap_cost_fp:.0f}×")
print(f"\n→ Missing a real gap is {config.gap_cost_fn / config.gap_cost_fp:.0f}× more costly than a false alarm")

## Section 2 — Requirement extraction evaluation

We define expert-labelled ground truth for 10 GDPR articles and compare
against the extractor's output. Metrics: precision, recall, F1.

In [ ]:
# ── Expert ground-truth: expected requirement count per article ──
expert_labels = {
    "Article 5":  {"expected_count": 6, "has_sub_items": True,  "has_conditions": False},
    "Article 6":  {"expected_count": 1, "has_sub_items": False, "has_conditions": True},
    "Article 7":  {"expected_count": 3, "has_sub_items": False, "has_conditions": False},
    "Article 13": {"expected_count": 6, "has_sub_items": True,  "has_conditions": False},
    "Article 15": {"expected_count": 2, "has_sub_items": False, "has_conditions": False},
    "Article 16": {"expected_count": 1, "has_sub_items": False, "has_conditions": False},
    "Article 17": {"expected_count": 3, "has_sub_items": True,  "has_conditions": True},
    "Article 20": {"expected_count": 2, "has_sub_items": False, "has_conditions": False},
    "Article 25": {"expected_count": 2, "has_sub_items": False, "has_conditions": False},
    "Article 32": {"expected_count": 4, "has_sub_items": True,  "has_conditions": False},
}

# ── Simulated extractor output (mirrors our corpus from nb 02) ──
extractor_output = {
    "Article 5":  {"extracted_count": 6, "got_sub_items": True,  "got_conditions": False},
    "Article 6":  {"extracted_count": 1, "got_sub_items": False, "got_conditions": True},
    "Article 7":  {"extracted_count": 3, "got_sub_items": False, "got_conditions": False},
    "Article 13": {"extracted_count": 6, "got_sub_items": True,  "got_conditions": False},
    "Article 15": {"extracted_count": 2, "got_sub_items": False, "got_conditions": False},
    "Article 16": {"extracted_count": 1, "got_sub_items": False, "got_conditions": False},
    "Article 17": {"extracted_count": 3, "got_sub_items": True,  "got_conditions": False},  # missed condition
    "Article 20": {"extracted_count": 2, "got_sub_items": False, "got_conditions": False},
    "Article 25": {"extracted_count": 2, "got_sub_items": False, "got_conditions": False},
    "Article 32": {"extracted_count": 4, "got_sub_items": True,  "got_conditions": False},
}

# ── Compute metrics ──
total_expected = sum(v["expected_count"] for v in expert_labels.values())
total_extracted = sum(v["extracted_count"] for v in extractor_output.values())

# Per-article TP/FP/FN
tp_total, fp_total, fn_total = 0, 0, 0
rows = []
for art in expert_labels:
    expected = expert_labels[art]["expected_count"]
    extracted = extractor_output[art]["extracted_count"]
    tp = min(expected, extracted)
    fp = max(0, extracted - expected)
    fn = max(0, expected - extracted)
    tp_total += tp
    fp_total += fp
    fn_total += fn

    # Check sub-items and conditions
    sub_ok = "✓" if extractor_output[art]["got_sub_items"] == expert_labels[art]["has_sub_items"] else "✗"
    cond_ok = "✓" if extractor_output[art]["got_conditions"] == expert_labels[art]["has_conditions"] else "✗"

    rows.append({
        "Article": art, "Expected": expected, "Extracted": extracted,
        "TP": tp, "FP": fp, "FN": fn, "Sub-items": sub_ok, "Conditions": cond_ok
    })

df_ext = pd.DataFrame(rows)
print("Requirement extraction evaluation")
print("=" * 80)
print(df_ext.to_string(index=False))

precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0
recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

print(f"\nAggregate: TP={tp_total}, FP={fp_total}, FN={fn_total}")
print(f"  Precision : {precision:.3f}")
print(f"  Recall    : {recall:.3f}")
print(f"  F1        : {f1:.3f}")

cond_accuracy = sum(1 for art in expert_labels
    if extractor_output[art]["got_conditions"] == expert_labels[art]["has_conditions"]) / len(expert_labels)
print(f"  Condition parsing accuracy: {cond_accuracy:.1%}")

## Section 3 — Coverage scoring accuracy

We create a **gold-standard dataset** of 30 requirement-policy pairs with
expert-assigned coverage scores (1-5). We measure:

- **Cohen's kappa** (inter-rater agreement between system and expert)
- **Mean Absolute Error** (how far off on average)
- **Confusion matrix** at the gap threshold (score < 3)

In [ ]:
# ── Gold-standard pairs (expert scores) ──
# Format: (requirement_id, brief_description, expert_score, system_score)
gold_standard = [
    ("GDPR-5-1a", "Lawful fair transparent processing", 4, 4),
    ("GDPR-5-1b", "Purpose limitation", 4, 3),
    ("GDPR-5-1c", "Data minimisation", 4, 4),
    ("GDPR-5-1d", "Accuracy", 4, 4),
    ("GDPR-5-1e", "Storage limitation", 5, 4),
    ("GDPR-5-1f", "Security principle", 5, 5),
    ("GDPR-6-1",  "Lawful basis", 3, 3),
    ("GDPR-7-1",  "Demonstrate consent", 2, 2),
    ("GDPR-7-2",  "Clear consent request", 2, 3),
    ("GDPR-7-3",  "Withdraw consent", 3, 3),
    ("GDPR-13-1a","Controller identity", 4, 4),
    ("GDPR-13-1b","DPO contact details", 1, 1),
    ("GDPR-13-1c","Processing purposes", 4, 4),
    ("GDPR-13-1d","Data recipients", 4, 5),
    ("GDPR-13-2a","Retention period info", 4, 4),
    ("GDPR-13-2b","Rights information", 4, 3),
    ("GDPR-15-1", "Right of access", 4, 4),
    ("GDPR-15-3", "Free copy of data", 3, 3),
    ("GDPR-16-1", "Right to rectification", 4, 4),
    ("GDPR-17-1", "Right to erasure", 5, 5),
    ("GDPR-17-1a","Erasure when unnecessary", 4, 4),
    ("GDPR-17-1b","Erasure on consent withdrawal", 3, 2),
    ("GDPR-20-1", "Data portability format", 1, 1),
    ("GDPR-20-2", "Portability to other controller", 1, 2),
    ("GDPR-25-1", "Privacy by design", 4, 3),
    ("GDPR-25-2", "Privacy by default", 3, 3),
    ("GDPR-32-1a","Encryption pseudonymisation", 5, 5),
    ("GDPR-32-1b","Confidentiality resilience", 4, 4),
    ("GDPR-32-1c","Restore availability", 4, 4),
    ("GDPR-32-1d","Security testing", 5, 5),
]

expert_scores = np.array([g[2] for g in gold_standard])
system_scores = np.array([g[3] for g in gold_standard])

# ── Cohen's kappa ──
kappa = cohen_kappa_score(expert_scores, system_scores, weights="linear")

# ── Mean Absolute Error ──
mae = mean_absolute_error(expert_scores, system_scores)

# ── Exact agreement ──
exact_match = np.mean(expert_scores == system_scores)

print("Coverage scoring calibration")
print("=" * 50)
print(f"  Cohen's κ (linear) : {kappa:.3f}  {'(substantial)' if kappa > 0.6 else '(moderate)'}")
print(f"  Mean Absolute Error: {mae:.3f}")
print(f"  Exact agreement    : {exact_match:.1%}")

# ── Score distribution comparison ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
for scores, label, color in [(expert_scores, "Expert", "#1976d2"), (system_scores, "System", "#f57c00")]:
    axes[0].hist(scores, bins=np.arange(0.5, 6.5, 1), alpha=0.6, label=label, color=color, edgecolor="white")
axes[0].set_xlabel("Coverage score")
axes[0].set_ylabel("Count")
axes[0].set_title("Score distribution: Expert vs System")
axes[0].legend()

# Scatter
axes[1].scatter(expert_scores, system_scores, alpha=0.6, s=80, edgecolors="white")
axes[1].plot([0, 6], [0, 6], "k--", alpha=0.3)
axes[1].set_xlabel("Expert score")
axes[1].set_ylabel("System score")
axes[1].set_title(f"Calibration (κ={kappa:.2f}, MAE={mae:.2f})")
axes[1].set_xlim(0.5, 5.5)
axes[1].set_ylim(0.5, 5.5)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrix at gap threshold ──
threshold = config.coverage_threshold
expert_gap = (expert_scores < threshold).astype(int)   # 1 = gap
system_gap = (system_scores < threshold).astype(int)

cm = confusion_matrix(expert_gap, system_gap, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print(f"Gap detection confusion matrix (threshold: score < {threshold})")
print(f"{'':15s} Predicted OK  Predicted Gap")
print(f"  Actual OK    {tn:>10}  {fp:>12}")
print(f"  Actual Gap   {fn:>10}  {tp:>12}")
print()
print(f"  True Positives  (correct gaps)  : {tp}")
print(f"  True Negatives  (correct OK)    : {tn}")
print(f"  False Positives (false alarms)  : {fp}")
print(f"  False Negatives (missed gaps)   : {fn}")

precision_gap = tp / (tp + fp) if (tp + fp) else 0
recall_gap = tp / (tp + fn) if (tp + fn) else 0
f1_gap = 2 * precision_gap * recall_gap / (precision_gap + recall_gap) if (precision_gap + recall_gap) else 0

print(f"\n  Gap precision: {precision_gap:.3f}")
print(f"  Gap recall   : {recall_gap:.3f}")
print(f"  Gap F1       : {f1_gap:.3f}")

## Section 4 — Gap detection cost analysis

Which error is worse?
- **False positive** (flagging a non-gap): wastes remediation effort (~$5 K)
- **False negative** (missing a real gap): regulatory exposure (~$50 K+)

We build a **cost matrix** and plot the **ROC curve** across different thresholds.

In [ ]:
# ── Cost analysis ──
fp_cost_total = fp * config.gap_cost_fp
fn_cost_total = fn * config.gap_cost_fn
total_cost = fp_cost_total + fn_cost_total

print("Gap detection cost analysis")
print("=" * 50)
print(f"  False positives  : {fp} × ${config.gap_cost_fp:,.0f} = ${fp_cost_total:,.0f}")
print(f"  False negatives  : {fn} × ${config.gap_cost_fn:,.0f} = ${fn_cost_total:,.0f}")
print(f"  Total error cost : ${total_cost:,.0f}")
print()
if fn > 0:
    print(f"  ⚠ {fn} missed gap(s) — each represents potential regulatory exposure")
else:
    print(f"  ✓ No missed gaps at threshold={threshold}")

# ── ROC curve across thresholds ──
# We vary the threshold from 1 to 5 and compute TPR/FPR
thresholds = [1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]
tpr_list, fpr_list, cost_list = [], [], []

for t in thresholds:
    sys_pred = (system_scores < t).astype(int)
    exp_true = (expert_scores < t).astype(int)

    tp_t = np.sum((sys_pred == 1) & (exp_true == 1))
    fp_t = np.sum((sys_pred == 1) & (exp_true == 0))
    fn_t = np.sum((sys_pred == 0) & (exp_true == 1))
    tn_t = np.sum((sys_pred == 0) & (exp_true == 0))

    tpr = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    fpr = fp_t / (fp_t + tn_t) if (fp_t + tn_t) > 0 else 0
    cost = fp_t * config.gap_cost_fp + fn_t * config.gap_cost_fn

    tpr_list.append(tpr)
    fpr_list.append(fpr)
    cost_list.append(cost)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC
axes[0].plot(fpr_list, tpr_list, "b-o", linewidth=2)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
for i, t in enumerate(thresholds):
    if t in (2, 3, 4):
        axes[0].annotate(f"t={t}", (fpr_list[i], tpr_list[i]),
                         textcoords="offset points", xytext=(8, -8), fontsize=8)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curve — gap detection across thresholds")

# Cost curve
axes[1].plot(thresholds, [c / 1000 for c in cost_list], "r-o", linewidth=2)
axes[1].set_xlabel("Gap threshold (score < t)")
axes[1].set_ylabel("Total error cost ($K)")
axes[1].set_title("Error cost by threshold")
axes[1].axvline(x=config.coverage_threshold, color="blue", linestyle="--", alpha=0.5, label=f"Current t={config.coverage_threshold}")
axes[1].legend()

plt.tight_layout()
plt.show()

optimal_t = thresholds[np.argmin(cost_list)]
print(f"\nOptimal threshold (min cost): {optimal_t}")
print(f"Current threshold: {config.coverage_threshold}")

## Section 5 — Remediation quality

We use **LLM-as-judge** to score 10 remediation samples on four criteria:

| Criterion | Description |
|-----------|-------------|
| **Specific** | Does it name exact policy sections and language? |
| **Actionable** | Can the compliance team implement it directly? |
| **Legally sound** | Is the recommended language legally appropriate? |
| **Complete** | Does it fully close the identified gap? |

Each criterion is scored 1-5. We compute an average quality score.

In [ ]:
# ── Sample remediations to evaluate ──
sample_remediations = [
    {"id": "GDPR-20-1", "gap": "No data portability mechanism",
     "remediation": "Add to Privacy Policy §4: 'Users may export all personal data in CSV or JSON format through Settings > Export Data. Export requests are processed within 24 hours.'"},
    {"id": "GDPR-20-2", "gap": "No transfer to other controller",
     "remediation": "Add to Privacy Policy §4: 'Users may request direct transfer of their data to another service provider. We support automated transfer via our API for controllers who implement the standard endpoint.'"},
    {"id": "GDPR-13-1b", "gap": "No DPO contact details",
     "remediation": "Add to Privacy Policy §1: 'Our Data Protection Officer can be reached at dpo@acmesaas.com or by post at Acme SaaS Corp, Attn: DPO, 100 Innovation Drive, San Francisco, CA 94105.'"},
    {"id": "GDPR-7-1", "gap": "Cannot demonstrate consent obtained",
     "remediation": "Implement consent logging: record timestamp, IP address, and version of consent text for each user. Add to Privacy Policy §6: 'We maintain records of each consent action including the date, method, and specific text agreed to.'"},
    {"id": "GDPR-7-2", "gap": "Consent not clearly distinguishable",
     "remediation": "Redesign registration flow: separate consent checkbox from Terms of Service. Add clear heading 'Data Processing Consent' with bullet-point summary of data uses before the checkbox."},
    {"id": "GDPR-25-1", "gap": "Privacy by design not specific",
     "remediation": "Add to Security Policy §7: 'All new features undergo a mandatory Privacy Impact Assessment (PIA) checklist before development begins. The PIA evaluates data minimisation, purpose limitation, and storage requirements.'"},
    {"id": "GDPR-5-1b", "gap": "Purpose limitation vague",
     "remediation": "Update Privacy Policy §3 to enumerate each processing purpose with its corresponding lawful basis in a table format."},
    {"id": "GDPR-15-3", "gap": "Data copy format not specified",
     "remediation": "Add to Privacy Policy §4: 'Data access requests are fulfilled in machine-readable JSON format. For human-readable reports, PDF format is also available upon request.'"},
    {"id": "GDPR-17-1b", "gap": "Erasure on consent withdrawal unclear",
     "remediation": "Add to Privacy Policy §4: 'When you withdraw consent, all data processed solely on the basis of that consent is deleted within 72 hours, unless another lawful basis applies.'"},
    {"id": "GDPR-25-2", "gap": "Privacy by default not demonstrated",
     "remediation": "Add to Security Policy: 'All new user accounts are created with the minimum data collection profile. Optional data fields are disabled by default. Analytics opt-in is off by default.'"},
]

JUDGE_PROMPT = """You are an expert compliance evaluator. Score this remediation on four criteria (1-5 each):

GAP: {gap}
REMEDIATION: {remediation}

Score each criterion 1-5:
- specific: Does it name exact policy sections and concrete language?
- actionable: Can the compliance team implement it directly without guesswork?
- legally_sound: Is the recommended language legally appropriate for GDPR?
- complete: Does it fully close the identified gap?

Respond in this exact JSON format:
{{"specific": <int>, "actionable": <int>, "legally_sound": <int>, "complete": <int>}}"""

results = []
for sample in sample_remediations:
    if client:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": JUDGE_PROMPT.format(
                gap=sample["gap"], remediation=sample["remediation"]
            )}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        scores = json.loads(resp.choices[0].message.content)
    else:
        # Deterministic mock based on remediation length as quality proxy
        base = min(5, max(3, len(sample["remediation"]) // 40))
        scores = {
            "specific": base,
            "actionable": min(5, base + 1),
            "legally_sound": base,
            "complete": max(2, base - 1),
        }

    scores["req_id"] = sample["id"]
    scores["avg"] = np.mean([scores["specific"], scores["actionable"],
                             scores["legally_sound"], scores["complete"]])
    results.append(scores)

df_qual = pd.DataFrame(results)
cols = ["req_id", "specific", "actionable", "legally_sound", "complete", "avg"]
df_qual = df_qual[cols]

print("Remediation quality assessment (LLM-as-judge)")
print("=" * 75)
print(df_qual.to_string(index=False, float_format="{:.1f}".format))

print(f"\nAverage quality scores:")
for dim in ["specific", "actionable", "legally_sound", "complete"]:
    avg = df_qual[dim].mean()
    print(f"  {dim:15s}: {avg:.2f}/5")
print(f"  {'overall':15s}: {df_qual['avg'].mean():.2f}/5")

In [ ]:
# ── Visualise remediation quality ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Radar-style bar chart
dimensions = ["specific", "actionable", "legally_sound", "complete"]
means = [df_qual[d].mean() for d in dimensions]
colors = ["#1976d2", "#388e3c", "#f57c00", "#7b1fa2"]

axes[0].barh(dimensions, means, color=colors, height=0.5)
axes[0].set_xlim(0, 5.5)
axes[0].set_xlabel("Average score (1-5)")
axes[0].set_title("Remediation quality by dimension")
for i, v in enumerate(means):
    axes[0].text(v + 0.1, i, f"{v:.2f}", va="center")

# Per-remediation heatmap
heat_data = df_qual[dimensions].values
im = axes[1].imshow(heat_data, cmap="RdYlGn", vmin=1, vmax=5, aspect="auto")
axes[1].set_xticks(range(len(dimensions)))
axes[1].set_xticklabels([d[:8] for d in dimensions], rotation=45)
axes[1].set_yticks(range(len(df_qual)))
axes[1].set_yticklabels(df_qual["req_id"].tolist(), fontsize=7)
axes[1].set_title("Quality heatmap per remediation")
plt.colorbar(im, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

## Section 6 — End-to-end analysis

How does the automated auditor compare to a manual audit?

In [ ]:
# ── Performance comparison ──
comparison = {
    "Metric": [
        "Time to complete audit",
        "Cost per audit",
        "Requirements analysed",
        "Coverage scoring accuracy",
        "Gap detection recall",
        "Remediation quality",
        "Continuous monitoring",
        "Multi-framework support",
    ],
    "Manual audit": [
        "4-12 weeks",
        "$50,000-$500,000",
        "~30 (GDPR subset)",
        "Expert judgement (κ ≈ 0.75 inter-rater)",
        "~85-95% (depends on auditor)",
        "High (but expensive)",
        "Annual / ad-hoc",
        "Separate engagement per framework",
    ],
    "AI auditor": [
        "< 5 minutes",
        "~$0.50 (API costs)",
        "30 (same GDPR subset)",
        f"κ = {kappa:.2f} vs expert",
        f"{recall_gap:.0%}",
        f"{df_qual['avg'].mean():.1f}/5 avg quality",
        "Real-time / daily",
        "Same pipeline, different corpus",
    ],
}

df_comp = pd.DataFrame(comparison)
print("Manual vs AI auditor comparison")
print("=" * 90)
print(df_comp.to_string(index=False, max_colwidth=40))

In [ ]:
# ── Failure mode analysis ──
failure_modes = [
    {
        "mode": "Ambiguous regulation language",
        "description": "Articles with vague or conditional language may be scored inconsistently",
        "frequency": "~10% of requirements",
        "mitigation": "Multi-pass assessment with different prompt framings, take conservative score",
    },
    {
        "mode": "Policy split across documents",
        "description": "A requirement satisfied by combining evidence from multiple policies",
        "frequency": "~15% of requirements",
        "mitigation": "Increase top_k retrieval, include cross-document evidence synthesis",
    },
    {
        "mode": "Implicit compliance",
        "description": "Company complies in practice but policy doesn't explicitly state it",
        "frequency": "~20% of gaps may be false positives",
        "mitigation": "Interview-augmented assessment, check implementation alongside policy",
    },
    {
        "mode": "Regulatory interpretation differences",
        "description": "Different jurisdictions interpret the same article differently",
        "frequency": "Varies by jurisdiction",
        "mitigation": "Jurisdiction-specific prompt templates, local legal review of results",
    },
    {
        "mode": "Stale policy documents",
        "description": "Policies don't reflect current practices after recent changes",
        "frequency": "Common in fast-growing companies",
        "mitigation": "Integrate with version control, alert when policy age > 6 months",
    },
]

print("Failure mode analysis")
print("=" * 80)
for fm in failure_modes:
    print(f"\n  Mode      : {fm['mode']}")
    print(f"  Frequency : {fm['frequency']}")
    print(f"  Mitigation: {fm['mitigation']}")

In [ ]:
# ── Final scorecard ──
print("╔" + "═" * 58 + "╗")
print("║           AUDITOR EVALUATION SCORECARD                  ║")
print("╠" + "═" * 58 + "╣")
print(f"║  Extraction F1          : {f1:.3f}                          ║")
print(f"║  Coverage Cohen's κ     : {kappa:.3f}                          ║")
print(f"║  Coverage MAE           : {mae:.3f}                          ║")
print(f"║  Gap detection precision: {precision_gap:.3f}                          ║")
print(f"║  Gap detection recall   : {recall_gap:.3f}                          ║")
print(f"║  Remediation quality    : {df_qual['avg'].mean():.1f}/5                           ║")
print(f"║  Error cost (30 reqs)   : ${total_cost:>8,.0f}                       ║")
print(f"║  Speed                  : <5 min vs 4-12 weeks            ║")
print(f"║  Cost                   : ~$0.50 vs $50K-$500K            ║")
print("╚" + "═" * 58 + "╝")
print()
print("Conclusion: The AI auditor achieves substantial agreement with expert")
print("judgement (κ > 0.6) at a fraction of the cost and time. Key improvement")
print("areas: reduce false negatives through multi-pass assessment, and improve")
print("remediation completeness for complex multi-document requirements.")

## Exercises

1. **Threshold optimisation** — Implement a grid search over gap thresholds (1.5 to
   4.5 in 0.5 steps) using the cost matrix. Plot cost vs. threshold and identify the
   optimal operating point for different FN/FP cost ratios (5×, 10×, 20×).

2. **Inter-model agreement** — If you have access to multiple LLMs (GPT-4o, Claude,
   Gemini), run the same 30 gold-standard pairs through each and compute pairwise
   Cohen's κ. Which models agree most? Which requirement types show the most disagreement?

3. **Bootstrapped confidence intervals** — Implement bootstrap resampling (1000 draws)
   on the 30 gold-standard pairs. Report 95% confidence intervals for Cohen's κ and MAE.
   How many gold-standard pairs would you need for ±0.05 precision on κ?

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Extraction achieves near-perfect recall but condition parsing needs improvement |
| 2 | Coverage scores show substantial agreement with experts (κ > 0.6) |
| 3 | Missing a real gap costs 10× more than a false alarm — tune thresholds accordingly |
| 4 | ROC analysis reveals optimal threshold may differ from the default |
| 5 | Remediation quality averages 3.5-4.5/5 — actionability is the strongest dimension |
| 6 | AI auditor is 1000× faster and 100×-1000× cheaper than manual audits |

## What's Next

You now have a complete, evaluated compliance auditor. Next steps:
- **Expand** to additional frameworks (SOC 2, HIPAA, ISO 27001)
- **Deploy** as a continuous monitoring service with regulation change alerts
- **Integrate** with document management for automatic policy updates

---

## References

1. Cohen, J. (1960). A coefficient of agreement for nominal scales. *Educational and Psychological Measurement*, 20(1), 37-46.
2. European Parliament, *Regulation (EU) 2016/679 — General Data Protection Regulation*, 2016.
3. Zheng, L. et al. (2023). Judging LLM-as-a-judge. *arXiv:2306.05685*.
4. GDPR Enforcement Tracker — <https://www.enforcementtracker.com/>